In [1]:
import requests
import configparser
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
from utils import plot_image, get_access_token
import numpy as np
from rasterio.io import MemoryFile
from datetime import datetime, timedelta
from sentinelhub import BBox, bbox_to_dimensions, CRS
from sentinelhub import SHConfig, SentinelHubCatalog, DataCollection
import evalscripts as eval
import pandas as pd
import os
import rasterio

/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [74]:
config_file = configparser.ConfigParser()
config_file.read("config.ini")

username = config_file["copernicus"]["username"]
password = config_file["copernicus"]["password"]

config = SHConfig()
config.sh_client_id = config_file["copernicus"]["client_id"] #"<CLIENT ID>"
config.sh_client_secret = config_file["copernicus"]["client_secret"] #<CLIENT SECRET>"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token" # Is it required?
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.save("cdse")
config = SHConfig("cdse")

In [9]:
def download_image_copernicus(access_token, time_interval, image_type, aoi, evalscript, resolution, config, save_path, cloud_cover_limit): 

    # Set the url to sent the request
    url = "https://sh.dataspace.copernicus.eu/api/v1/process"

    # Define the Area of Interest
    aoi_bbox = BBox(bbox=aoi, crs=CRS.WGS84)
    aoi_size = bbox_to_dimensions(aoi_bbox, resolution=resolution)

    # Use SentinelHubCatalog to check if there are any images in the time interval of interest
    # If there are, extract the date (can't get it directly from the tiff)
    catalog = SentinelHubCatalog(config=config)
    date_range = time_interval[0],  time_interval[1]
    search_iterator = catalog.search(
        DataCollection.SENTINEL2_L2A,
        bbox=aoi_bbox,
        time= date_range ,
        fields={"include": ["id", "properties.eo:cloud_cover"], "exclude": [ "properties.datetime"]},
    )
    results = list(search_iterator)
    unique_results = {}
    # A partir del id guardamos las fechas
    for item in results:
        acquisition_id = item['id'].split('_T')[0]
        if acquisition_id not in unique_results:
            unique_results[acquisition_id] = item
    unique_results = list(unique_results.values())
    ids = [item['id'] for item in unique_results]
    dates = [datetime.strptime(id.split('_')[2][:8], "%Y%m%d").date() for id in ids]
    
    # Guardamos cloud cover de las imágenes en el intervalo
    cloud_covers = []
    for item in results:
        cloud_cover = item["properties"]["eo:cloud_cover"]
        cloud_covers.append(cloud_cover)

    if len(dates) == 0: 
        print(f"Request empty for dates {time_interval}")
        return None, None
    
    # Comprobar si la imagen está cubierta por nubes antes de seguir
    elif any(cc > cloud_cover_limit for cc in cloud_covers):
        print(f"Request covered by clouds for dates {time_interval}, with coverage {cloud_covers}")
        return None, None       

    else: 
        date_string = [d.isoformat() for d in dates][0]

        # Define the content for the post
        headers={
        "Content-Type": "application/json",
        "Authorization" : "Bearer "+ access_token
        }


        json={
            "input": {
                "bounds": {
                    "bbox": aoi
                },
                "data": [
                    {
                    "dataFilter": {
                        "timeRange": {
                        "from": time_interval[0] + "T00:00:00Z", #"2024-10-12T00:00:00Z", #YYYY-mm-dd
                        "to": time_interval[1] + "T23:59:59Z"#"2024-11-12T23:59:59Z"
                        },
                        "mosaickingOrder": "leastCC"
                    
                    },
                    "type": "sentinel-2-l2a"
                    }]
            },
            "output": {
                "width": aoi_size[0],#1271,
                "height": aoi_size[1],#2183

                "responses": [
                    {
                        "format": {
                            "type": "image/" + image_type
                        }
                    }
                ]
            },
            "evalscript" : evalscript,
            "data_folder" : "test_dir",
            "save_data" : True
        }

        response = requests.post(url, headers=headers, json=json)

        if response.status_code == 200:
            print(f"Request completed for date {date_string} in interval {time_interval}")
            
            if save_path:
                file_name = f"{date_string}.tiff"
                file_path = os.path.join(save_path, file_name)

                with MemoryFile(response.content) as memfile:
                    with memfile.open() as dataset:
                        with rasterio.open(file_path, 'w', **dataset.profile) as dst:
                            dst.write(dataset.read())
                print(f"Saved TIFF file to {file_path}")
            
            return response, date_string
        else:
            print(f"Request failed {response.content}")
            return None, None

In [38]:
evalscript_all_bands = """
    //VERSION=3
    function setup() {
        return {
            input: [{
                bands: ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B11","B12"],
                units: ["reflectance","reflectance","reflectance","reflectance","reflectance","reflectance","reflectance","reflectance","reflectance","reflectance","reflectance","reflectance"]
            }],
            output: {
                bands: 12,
                sampleType: "UINT16"
            }
        };
    }

    function evaluatePixel(sample) {
        return [sample.B01,
                sample.B02,
                sample.B03,
                sample.B04,
                sample.B05,
                sample.B06,
                sample.B07,
                sample.B08,
                sample.B8A,
                sample.B09,
                sample.B11,
                sample.B12];
    }
"""

In [145]:
evalscript_all_bands = """
    //VERSION=3
    function setup() {
        return {
            input: [{
                bands: ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B11","B12", "SCL"]
            }],
            output: {
                bands: 13,
                sampleType: "UINT16"
            }
        };
    }

    function evaluatePixel(sample) {
        return [sample.B01,
                sample.B02,
                sample.B03,
                sample.B04,
                sample.B05,
                sample.B06,
                sample.B07,
                sample.B08,
                sample.B8A,
                sample.B09,
                sample.B11,
                sample.B12,
                sample.SCL];
    }
"""

In [146]:
access_token = get_access_token(username, password)
evalscript = evalscript_all_bands
image_type = "tiff" 
aoi = [-0.866977, 37.628916, -0.71696, 37.822802]
aoi = [-0.86, 37.65, -0.74, 37.8]
resolution = 10
cloud_cover_limit = 75
## Time interval ## 
year = 2021
start = datetime(year, 1, 9)
end = datetime(year, 1, 11)
tdelta = timedelta(days=4)
n_chunks = round((end - start)/tdelta)
starts = [(start + i * tdelta).date().isoformat() for i in range(n_chunks+1)]
ends = [(start + timedelta(days=4) + i * tdelta).date().isoformat() for i in range(n_chunks+1)]
slots = [(starts[i], ends[i]) for i in range(len(starts))]
folder = "test_dir"

In [11]:
# Slots que nos interesan de las fechas que están sincronizadas
from datetime import datetime, timedelta

target_dates = [
    "2017-06-30", "2018-01-31", "2018-02-20", "2018-03-07", "2018-05-11", "2018-05-16",
    "2018-06-20", "2018-07-10", "2018-08-09", "2018-08-14", "2018-08-29", "2018-10-03",
    "2018-11-07", "2019-02-20", "2019-03-12", "2019-06-25", "2019-07-10", "2019-08-14",
    "2019-09-18", "2019-10-03", "2019-11-27", "2020-02-20", "2020-02-25", "2020-03-11",
    "2020-05-05", "2020-05-20", "2020-08-13", "2020-12-21", "2021-01-05", "2021-04-20", 
    "2021-06-14", "2021-07-14", "2021-08-03", "2021-08-13", "2021-11-11", "2021-12-01",
    "2022-02-24", "2022-06-24", "2022-07-14", "2022-08-03", "2022-09-07", "2023-01-10", 
    "2023-01-20", "2023-03-01", "2023-03-16", "2023-04-20", "2023-05-25", "2023-07-19", 
    "2023-09-07", "2023-09-27", "2023-11-16", "2024-04-24", "2024-05-29", "2024-06-18", 
    "2024-07-03", "2024-07-18"
]

slots = [((datetime.strptime(d, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d"), (datetime.strptime(d, "%Y-%m-%d") + timedelta(days=4)).strftime("%Y-%m-%d")) for d in target_dates]
slots

[('2017-06-29', '2017-07-04'),
 ('2018-01-30', '2018-02-04'),
 ('2018-02-19', '2018-02-24'),
 ('2018-03-06', '2018-03-11'),
 ('2018-05-10', '2018-05-15'),
 ('2018-05-15', '2018-05-20'),
 ('2018-06-19', '2018-06-24'),
 ('2018-07-09', '2018-07-14'),
 ('2018-08-08', '2018-08-13'),
 ('2018-08-13', '2018-08-18'),
 ('2018-08-28', '2018-09-02'),
 ('2018-10-02', '2018-10-07'),
 ('2018-11-06', '2018-11-11'),
 ('2019-02-19', '2019-02-24'),
 ('2019-03-11', '2019-03-16'),
 ('2019-06-24', '2019-06-29'),
 ('2019-07-09', '2019-07-14'),
 ('2019-08-13', '2019-08-18'),
 ('2019-09-17', '2019-09-22'),
 ('2019-10-02', '2019-10-07'),
 ('2019-11-26', '2019-12-01'),
 ('2020-02-19', '2020-02-24'),
 ('2020-02-24', '2020-02-29'),
 ('2020-03-10', '2020-03-15'),
 ('2020-05-04', '2020-05-09'),
 ('2020-05-19', '2020-05-24'),
 ('2020-08-12', '2020-08-17'),
 ('2020-12-20', '2020-12-25'),
 ('2021-01-04', '2021-01-09'),
 ('2021-04-19', '2021-04-24'),
 ('2021-06-13', '2021-06-18'),
 ('2021-07-13', '2021-07-18'),
 ('2021-

In [152]:
for time_interval in slots:
    # Get the satellite response for the current time interval
    response, date_taken = download_image_copernicus(access_token, time_interval, image_type, aoi, evalscript, resolution, config, folder, cloud_cover_limit)
    if response is None:
        print(f"No data available for time interval {time_interval}")
        continue 

    if response.status_code == 200:
        print("Ok")

Request failed b'{"error":{"status":500,"reason":"Internal Server Error","message":"java.util.concurrent.ExecutionException: java.lang.RuntimeException: Illegal request! URL: creo://sh_idx_s2l2a_2021_01/Sentinel-2/MSI/L2A_N0500/2021/01/10/S2B_MSIL2A_20210110T105329_N0500_R051_T30SXG_20230531T071634.SAFE/GRANULE/L2A_T30SXG_A020097_20210110T105325/IMG_DATA/R60m/T30SXG_20210110T105329_B09_60m.json.gz status: 503 reason: Slow Down","code":"RENDERER_EXCEPTION"}}'
No data available for time interval ('2021-01-09', '2021-01-13')
